In [ ]:
!pip install unsloth

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import subprocess
import polars as pl
import torch
from google.colab import userdata  # Pro Google Colab
from kaggle_secrets import UserSecretsClient  # Pro Kaggle
from huggingface_hub import login, HfApi, DatasetCard, DatasetCardData
from datasets import load_dataset
from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer
from unsloth import FastLanguageModel
from tqdm import tqdm

### **1. LOAD SECRETS (Kaggle/Colab Fallback) & HF LOGIN**

In [ ]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    user_email = user_secrets.get_secret("user_email")
    user_name = user_secrets.get_secret("user_name")
except Exception:
    try:
        hf_token = userdata.get("HF_TOKEN")
        user_email = userdata.get("USER_EMAIL")
        user_name = userdata.get("USER_NAME")
    except Exception as e:
        print(f"Warning: Failed to load secrets automatically: {e}")

In [ ]:
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    raise ValueError("HF_TOKEN was not found in Secrets!")

### **2. GIT CONFIGURATION***

In [ ]:
def set_git_config(email, name):
    try:
        subprocess.run(
            ["git", "config", "--global", "user.email", email], check=True
        )
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git configuration set for: {name} <{email}>")
    except subprocess.CalledProcessError as e:
        print(f"Error setting Git configuration: {e}")

In [ ]:
if user_email and user_name:
    set_git_config(user_email, user_name)

### **3. SETUP REPO_ID AND INITIALIZATION**

In [ ]:
api = HfApi()
repo_exists = False
hf_username = "KRadim"

In [ ]:
DATASET_REPO_ID = f"{hf_username}/multilingual-text2cypher-2025"

### **4. LOADING AND MERGING THE ORIGINAL DATASET (TRAIN + TEST) INTO POLARS**

In [ ]:
ds = load_dataset("neo4j/text2cypher-2025v1")
df_raw_train = pl.from_pandas(ds["train"].to_pandas())

In [ ]:
if "test" in ds:
    df_raw_test = pl.from_pandas(ds["test"].to_pandas())
    df_raw = pl.concat([df_raw_train, df_raw_test])
    print(f"Merged! Train: {len(df_raw_train)}, Test: {len(df_raw_test)} | Total to process: {len(df_raw)}")
else:
    df_raw = df_raw_train
    print(f"There is no test split in the dataset; I am proceeding only with the train split ({len(df_raw)} rows).")

### **5. MODEL LOADING AND BATCH TRANSLATION**

In [ ]:
max_seq_length = 2048  # Maximum context length
model_name = "google/gemma-4-E4B-it"

In [ ]:
print(f"\n--> [2/6] Loading model {model_name} onto GPU...")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
# 2. Activating fast inference mode
FastLanguageModel.for_inference(model)

In [ ]:
BATCH_SIZE = 8

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "left"

In [ ]:
def translate_batch(questions: list[str], target_language: str) -> list[str]:
    prompts = []
    for q in questions:
        messages = [
            {
                "role": "system",
                "content": (
                    f"You are a professional technical translator. Translate the following user question "
                    f"into {target_language}. Maintain exact technical terms, entity names, and schema context. "
                    f"Provide ONLY the direct translation, without any introduction or markdown formatting."
                )
            },
            {"role": "user", "content": q}
        ]
            # Skip_multimodal_checks=True prevents false Base64 detections in text
        prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True,
            skip_multimodal_checks=True
        )
        prompts.append(prompt)

    inputs = tokenizer(
        prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    translations = []
    for i, out in enumerate(outputs):
        input_len = inputs["input_ids"][i].shape[0]
        gen_tokens = out[input_len:]
        translated_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        translations.append(translated_text)

    return translations

In [ ]:
TARGET_LANGUAGES = {
    "question_cz": "Czech",
    "question_it": "Italian",
    "question_de": "German",
    "question_es": "Spanish",
    "question_fr": "French"
}

In [ ]:
questions_en = df_raw["question"].to_list()
translated_columns = {}

In [ ]:
print("\n--> [3/6] Generating multilingual queries...")
for col_name, lang_name in TARGET_LANGUAGES.items():
    print(f"Generating mutation: {lang_name} ({col_name})...")
    translated_list = []
    for i in tqdm(range(0, len(questions_en), BATCH_SIZE)):
        batch = questions_en[i : i + BATCH_SIZE]
        translated_batch = translate_batch(batch, target_language=lang_name)
        translated_list.extend(translated_batch)
    translated_columns[col_name] = translated_list

### **6. UNPIVOT (MELT), SHUFFLE, AND A NEW TRAIN/TEST SPLIT IN POLARS**

In [ ]:
print("\n--> [4/6] Transforming data in Polars (Unpivot/Melt + Shuffle)...")

In [ ]:
df_multilingual = df_raw.rename({"question": "question_en"})

In [ ]:
for col_name, values in translated_columns.items():
    df_multilingual = df_multilingual.with_columns(
        pl.Series(name=col_name, values=values)
    )

In [ ]:
# 💾 SAVE HERE: Save intermediate result to disk in case of a crash
df_multilingual.write_parquet("multilingual_checkpoint.parquet")
print("✅ Intermediate result successfully saved to 'multilingual_checkpoint.parquet'")

#### *Converting question columns into separate rows*

In [ ]:
df_flat = df_multilingual.unpivot(
    on=["question_en", "question_cz", "question_it", "question_de", "question_es", "question_fr"],
    index=["cypher", "schema", "data_source", "instance_id"],
    variable_name="language",
    value_name="question"
)

In [ ]:
df_flat = df_flat.with_columns(
    pl.col("language").str.replace("question_", "")
)

#### *Data shuffling for multilingual training*

In [ ]:
df_final = df_flat.sample(fraction=1.0, shuffle=True, seed=42)

In [ ]:
print("\n--> [5/6] I am splitting the complete multilingual dataset into a new Train (90%) and Test (10%) set...")
test_size = int(len(df_final) * 0.10)
df_test = df_final.head(test_size)
df_train = df_final.tail(len(df_final) - test_size)

In [ ]:
print(f"Total rows after unpivot: {len(df_final)} | New Train: {len(df_train)} | New Test: {len(df_test)}")

### **7. UPLOADING THE DATASET AND DATASET CARD (README) TO THE HF HUB**

In [ ]:
print(f"\n--> [6/6] Uploading dataset to repository: {DATASET_REPO_ID}...")

In [ ]:
dataset_dict = DatasetDict({
    "train": Dataset.from_polars(df_train),
    "test": Dataset.from_polars(df_test)
})

#### *Pushing the dataset to the HF Hub*

In [ ]:
dataset_dict.push_to_hub(
    repo_id=DATASET_REPO_ID,
    token=hf_token,
    private=False
)

In [ ]:
print("\n--> [7/6] Creating and uploading Dataset Card (README.md)...")

#### *Dataset card metadata*

In [ ]:
card_metadata = DatasetCardData(
    language=["en", "cs", "it", "de", "es", "fr"],
    license="apache-2.0",
    tags=["text-to-cypher", "neo4j", "multilingual", "synthetic", "gemma-4", "graph-db"],
    dataset_info={
        "features": [
            {"name": "language", "dtype": "string"},
            {"name": "question", "dtype": "string"},
            {"name": "cypher", "dtype": "string"},
            {"name": "schema", "dtype": "string"},
            {"name": "data_source", "dtype": "string"},
            {"name": "instance_id", "dtype": "string"},
        ],
        "splits": [
            {"name": "train", "num_bytes": dataset_dict["train"].size_in_bytes, "num_examples": len(dataset_dict["train"])},
            {"name": "test", "num_bytes": dataset_dict["test"].size_in_bytes, "num_examples": len(dataset_dict["test"])},
        ],
    }
)

#### *Dataset Card Text Description*

In [ ]:
dataset_card_text = f"""
# Multilingual Text-to-Cypher Dataset (2025)

This dataset is an expanded, **multilingual synthetic edition** created by combining all original splits of [`neo4j/text2cypher-2025v1`](https://huggingface.co/datasets/neo4j/text2cypher-2025v1).

It is designed for fine-tuning Large Language Models (LLMs) to generate accurate **Neo4j Cypher queries** from natural language prompts across 6 languages:
- 🇬🇧 **English (`en`)**
- 🇨🇿 **Czech (`cz`)**
- 🇮🇹 **Italian (`it`)**
- 🇩🇪 **German (`de`)**
- 🇪🇸 **Spanish (`es`)**
- 🇫🇷 **French (`fr`)**

---

## Dataset Splits

- **`train`**: {len(dataset_dict['train'])} examples (90%)
- **`test`**: {len(dataset_dict['test'])} examples (10%)

---

## Dataset Structure

All original data splits were combined, translated into 5 additional target languages, unpivoted, and globally reshuffled into balanced train/test splits.

### Fields:
- `language`: Language code (`en`, `cz`, `it`, `de`, `es`, `fr`).
- `question`: Natural language question in the specified language.
- `cypher`: Corresponding Neo4j Cypher query.
- `schema`: Database graph schema definition.
- `data_source`: Source database reference.
- `instance_id`: Unique identifier for the base prompt.

---

## Creation Process

1. **Data Concatenation**: Combined all split data from `neo4j/text2cypher-2025v1`.
2. **Translation & Synthesis**: Translated queries into CZ, IT, DE, ES, and FR using **`gemma-4-E4B-it`**.
3. **Data Transformation**: Unpivoted using **Polars** (expanding row count 6x).
4. **Reshuffling & Splitting**: Globally shuffled with seed `42` and split into new `train` (90%) and `test` (10%) splits.

---

## Usage in Python / Hugging Face

```python
from datasets import load_dataset

dataset = load_dataset("{DATASET_REPO_ID}")

train_data = dataset["train"]
test_data = dataset["test"]

print("Train sample:", train_data[0])
print("Test sample:", test_data[0])
"""

In [ ]:
card = DatasetCard(dataset_card_text)
card.data = card_metadata
card.push_to_hub(repo_id=DATASET_REPO_ID, token=hf_token)

In [ ]:
print("\n" + "=" * 60)
print(f"✅ Everything successfully completed!")
print(f"The dataset is available at: https://huggingface.co/datasets/{DATASET_REPO_ID}")
print("=" * 60)